In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import matplotlib.pyplot as plt
from wko5.training_load import build_pmc, current_fitness, ef_trend
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## Performance Management Chart

In [ ]:
pmc = build_pmc()
if not pmc.empty:
    fig, ax1 = plt.subplots(figsize=(16, 6))
    ax1.fill_between(pmc['date'], pmc['CTL'], alpha=0.3, color='blue', label='CTL (Fitness)')
    ax1.fill_between(pmc['date'], pmc['ATL'], alpha=0.3, color='red', label='ATL (Fatigue)')
    ax1.plot(pmc['date'], pmc['TSB'], color='gold', linewidth=1, label='TSB (Form)')
    ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Training Load')
    ax1.set_title('Performance Management Chart')
    ax1.legend()
    plt.tight_layout()
    plt.show()

## Current Fitness

In [ ]:
fitness = current_fitness()
print(f"As of {fitness.get('date', 'N/A')}:")
print(f"  CTL (Fitness): {fitness['CTL']}")
print(f"  ATL (Fatigue): {fitness['ATL']}")
print(f"  TSB (Form):    {fitness['TSB']}")

## Weekly TSS (Last 12 Months)

In [ ]:
if not pmc.empty:
    recent = pmc[pmc['date'] >= pmc['date'].max() - pd.Timedelta(days=365)]
    weekly = recent.groupby(recent['date'].dt.to_period('W'))['TSS'].sum()
    fig, ax = plt.subplots(figsize=(16, 4))
    weekly.plot(kind='bar', ax=ax, color='steelblue', alpha=0.7)
    ax.set_xlabel('Week')
    ax.set_ylabel('Weekly TSS')
    ax.set_title('Weekly Training Stress (Last 12 Months)')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

## Efficiency Factor Trend

In [ ]:
ef = ef_trend(days=365)
if not ef.empty:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.scatter(pd.to_datetime(ef['date']), ef['EF'], alpha=0.5, s=20, color='green')
    ef_sorted = ef.sort_values('date')
    ef_sorted['EF_rolling'] = ef_sorted['EF'].rolling(window=10, min_periods=3).mean()
    ax.plot(pd.to_datetime(ef_sorted['date']), ef_sorted['EF_rolling'], color='darkgreen', linewidth=2)
    ax.set_xlabel('Date')
    ax.set_ylabel('Efficiency Factor (NP/HR)')
    ax.set_title('Aerobic Efficiency Trend')
    plt.tight_layout()
    plt.show()